In [1]:
import torch
import random
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score

import torch.nn.functional as F

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
random.seed(seed)
df = pd.read_csv("./data/TE/TEP.csv", header=None)
df.iloc[df.iloc[:, 9] >= 0.612558, 9] = 3
df.iloc[(df.iloc[:, 9] < 0.612558) & (df.iloc[:, 9] >= 0.580210), 9] = 2
df.iloc[(df.iloc[:, 9] < 0.580210) & (df.iloc[:, 9] >= 0.555355), 9] = 1
df.iloc[(df.iloc[:, 9] < 0.555355), 9] = 0

scalar = StandardScaler()
df.iloc[:, :-1] = scalar.fit_transform(df.iloc[:, :-1])
npdata = df.values.astype(np.float32)
TE_dataset = torch.utils.data.TensorDataset(
    torch.tensor(npdata[:, :-1]), torch.tensor(npdata[:, -1]).to(torch.int64)
)
idx = [i for i in range(len(TE_dataset))]
random.shuffle(idx)
train_data = torch.utils.data.Subset(TE_dataset, idx[:1200])
val_data = torch.utils.data.Subset(TE_dataset, idx[1200:1600])
test_data = torch.utils.data.Subset(TE_dataset, idx[1600:])
train_loader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_data, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=64, shuffle=True)

In [2]:
def ts_append(a, b):
    """List like 'Append' tool for tensor datatype
    ---
    Parameters:
        a, b: append a with b
    """
    if a is None:
        return b
    else:
        return torch.cat([a, b], dim=0)

In [3]:
def valid(net, valid_data):
    yAll = None
    outAll = None
    outProb = None

    with torch.no_grad():
        for data in valid_data:
            x, y = data
            x = x.cuda()
            y = y.cuda()

            output = net(x)
            outAll = ts_append(outAll, output.argmax(1))
            outProb = ts_append(outProb, output)
            yAll = ts_append(yAll, y)

    acc = outAll.eq(yAll).float().mean().item()
    f1_mac = f1_score(outAll.cpu().numpy(), yAll.cpu().numpy(), average="macro")
    f1_mic = f1_score(outAll.cpu().numpy(), yAll.cpu().numpy(), average="micro")
    auc_s = roc_auc_score(yAll.cpu().numpy(), outProb.cpu().numpy(), multi_class="ovo")
    return (acc, f1_mac, f1_mic, auc_s)

### MLP

In [4]:
import torch.nn as nn


class MLPNet(nn.Module):
    def __init__(self, in_features, hidden_features, out_features):
        super(MLPNet, self).__init__()
        self.model = nn.ModuleList(
            [
                nn.Linear(in_features, hidden_features),
                nn.Linear(hidden_features, out_features),
            ]
        )
        self.relu = nn.ReLU()

    def forward(self, input):
        hidden = self.relu(self.model[0](input))
        return F.softmax(self.model[1](hidden), dim=1)

In [ ]:
from tqdm import tqdm

lr = 1e-2
wd = 1e-5
epoch = 1000

model = MLPNet(9, 64, 4).cuda()
opt = torch.optim.SGD(model.parameters(), lr=lr)

pbar = tqdm(total=epoch)
for i in range(epoch):
    pbar.set_description_str(f"Epoch: {i}/{epoch}")
    total_loss = 0
    for inputs, labels in train_loader:
        inputs = inputs.cuda()
        labels = labels.cuda()
        opt.zero_grad()

        output = model(inputs)
        loss = F.nll_loss(output, labels)
        loss.backward()
        opt.step()

        total_loss += loss.item()

    trn_acc, _, _, _ = valid(model, train_loader)
    val_acc, _, _, _ = valid(model, val_loader)
    if i % 1 == 0:
        test_acc, _, _, _ = valid(model, test_loader)

    total_loss = total_loss / len(train_loader)
    pbar.set_postfix(
        loss=total_loss, trn_acc=trn_acc, val_acc=val_acc, test_acc=test_acc
    )
    pbar.update(1)

with torch.no_grad():
    test_acc, test_f1_mac, test_f1_mic, test_auc = valid(model, test_loader)
print(
    "test_acc:",
    test_acc,
    "f1 macro:",
    test_f1_mac,
    "f1 micro",
    test_f1_mic,
    "auc",
    test_auc,
)
pbar.close()

Epoch: 999/1000: 100%|██████████| 1000/1000 [00:28<00:00, 35.19it/s, loss=-0.789, test_acc=0.815, trn_acc=0.818, val_acc=0.777]

test_acc: 0.8149999976158142 f1 macro: 0.8137034659820283 f1 micro 0.815 auc 0.9530627107651735


In [35]:
def label_gen_x(x, label):
    label = F.one_hot(
        torch.tensor(label).unsqueeze(0).repeat(x.shape[0]), num_classes=4
    ).cuda()
    return torch.cat([x, label], dim=1)


def pos_neg_gen(x, y):
    y_one_hot = F.one_hot(y, num_classes=4)
    y_rand = torch.randint(1, 3, (y.shape[0],)).cuda()
    y_rand = (y + y_rand) % 4
    y_rand_one_hot = F.one_hot(y_rand, num_classes=4)

    return torch.cat([x, y_one_hot], dim=1), torch.cat([x, y_rand_one_hot], dim=1)

### Forward forward algorithm

In [9]:
class FFNet(torch.nn.Module):

    def __init__(self, dims):
        super().__init__()
        self.layers = []
        for d in range(len(dims) - 1):
            self.layers += [Layer(dims[d], dims[d + 1]).cuda()]

    def forward(self, x):
        goodness_per_label = []
        for label in range(4):
            h = label_gen_x(x, label)
            goodness = []
            for layer in self.layers:
                h = layer(h)
                goodness += [h.pow(2).mean(1)]
            goodness_per_label += [sum(goodness).unsqueeze(1)]
        goodness_per_label = torch.cat(goodness_per_label, 1)
        return F.softmax(goodness_per_label, dim=1)

    def train(self, x_pos, x_neg):
        h_pos, h_neg = x_pos, x_neg
        for i, layer in enumerate(self.layers):
            print("training layer", i, "...")
            h_pos, h_neg = layer.train(h_pos, h_neg)


class Layer(nn.Linear):
    def __init__(self, in_features, out_features, bias=True, device=None, dtype=None):
        super().__init__(in_features, out_features, bias, device, dtype)
        self.relu = torch.nn.ReLU()
        self.opt = torch.optim.Adam(self.parameters(), lr=0.03)
        self.threshold = 2.0
        self.num_epochs = 1000

    def forward(self, x):
        x_direction = x / (x.norm(2, 1, keepdim=True) + 1e-4)
        return self.relu(torch.mm(x_direction, self.weight.T) + self.bias.unsqueeze(0))

    def train(self, x_pos, x_neg):
        for i in tqdm(range(self.num_epochs)):
            g_pos = self.forward(x_pos).pow(2).mean(1)
            g_neg = self.forward(x_neg).pow(2).mean(1)
            # The following loss pushes pos (neg) samples to
            # values larger (smaller) than the self.threshold.
            loss = torch.log(
                1
                + torch.exp(
                    torch.cat([-g_pos + self.threshold, g_neg - self.threshold])
                )
            ).mean()
            self.opt.zero_grad()
            # this backward just compute the derivative and hence
            # is not considered backpropagation.
            loss.backward()
            self.opt.step()
        return self.forward(x_pos).detach(), self.forward(x_neg).detach()

In [10]:
ff_net = FFNet([13, 2000, 2000])

ff_train_loader = torch.utils.data.DataLoader(train_data, batch_size=1200, shuffle=True)
ff_val_loader = torch.utils.data.DataLoader(val_data, batch_size=400, shuffle=True)
ff_test_loader = torch.utils.data.DataLoader(test_data, batch_size=400, shuffle=True)

x, y = next(iter(ff_train_loader))
x, y = x.cuda(), y.cuda()
x_pos, x_neg = pos_neg_gen(x, y)

ff_net.train(x_pos, x_neg)

val_x, val_y = next(iter(ff_val_loader))
val_x, val_y = val_x.cuda(), val_y.cuda()

test_x, test_y = next(iter(ff_test_loader))
test_x, test_y = test_x.cuda(), test_y.cuda()


test_acc, test_f1_mac, test_f1_mic, test_auc = valid(ff_net, ff_test_loader)
print(
    "test_acc:",
    test_acc,
    "f1 macro:",
    test_f1_mac,
    "f1 micro",
    test_f1_mic,
    "auc",
    test_auc,
)

training layer 0 ...


  0%|          | 0/1000 [00:00<?, ?it/s]

100%|██████████| 1000/1000 [00:00<00:00, 1932.28it/s]


training layer 1 ...


100%|██████████| 1000/1000 [00:01<00:00, 784.60it/s]

test_acc: 0.5149999856948853 f1 macro: 0.4455395398812784 f1 micro 0.515 auc 0.8327936711248337


In [21]:
def valid_no_model(output, y):
    acc = sum(output == y) / (output.shape[0])
    f1_mac = f1_score(output, y, average="macro")
    f1_mic = f1_score(output, y, average="micro")
    auc_s = roc_auc_score(output, F.one_hot(torch.tensor(y)).numpy(), multi_class="ovo")
    return (acc, f1_mac, f1_mic, auc_s)

### KNN

In [22]:
import numpy as np
from sklearn.neighbors import (
    KNeighborsClassifier,
)
from sklearn.metrics import classification_report

trn_X, trn_Y = next(iter(ff_train_loader))
trn_X, trn_Y = trn_X.numpy(), trn_Y.numpy()

val_X, val_Y = next(iter(ff_val_loader))
val_X, val_Y = val_X.numpy(), val_Y.numpy()

test_X, test_Y = next(iter(ff_test_loader))
test_X, test_Y = test_X.numpy(), test_Y.numpy()

knn = KNeighborsClassifier(n_neighbors=4)
knn.fit(trn_X, trn_Y)

val_out = knn.predict(val_X)
test_out = knn.predict(test_X)

test_acc, test_f1_mac, test_f1_mic, test_auc = valid_no_model(test_out, test_Y)
print(
    "test_acc:",
    test_acc,
    "f1 macro:",
    test_f1_mac,
    "f1 micro",
    test_f1_mic,
    "auc",
    test_auc,
)

test_acc: 0.7375 f1 macro: 0.7328486139391991 f1 micro 0.7375 auc 0.8285842965420357


### KMeans

In [31]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=4, random_state=56)
kmeans.fit(test_X)

# print(sum(test_Y == kmeans.labels_) / test_Y.shape[0])
test_acc, test_f1_mac, test_f1_mic, test_auc = valid_no_model(kmeans.labels_, test_Y)
print(
    "test_acc:",
    test_acc,
    "f1 macro:",
    test_f1_mac,
    "f1 micro",
    test_f1_mic,
    "auc",
    test_auc,
)

test_acc: 0.3975 f1 macro: 0.3454833729018976 f1 micro 0.3975 auc 0.554975989402219


### Decision Trees

In [33]:
from sklearn import tree

d_tree = tree.DecisionTreeClassifier()
d_tree = d_tree.fit(trn_X, trn_Y)

# trn_acc = d_tree.score(trn_X, trn_Y)

# val_acc = d_tree.score(val_X, val_Y)

# test_acc = d_tree.score(test_X, test_Y)

test_acc, test_f1_mac, test_f1_mic, test_auc = valid_no_model(
    d_tree.predict(test_X), test_Y
)
print(
    "test_acc:",
    test_acc,
    "f1 macro:",
    test_f1_mac,
    "f1 micro",
    test_f1_mic,
    "auc",
    test_auc,
)

test_acc: 0.7725 f1 macro: 0.7635249807628244 f1 micro 0.7725 auc 0.8423773330688226


### Naive Bayesian

In [34]:
from sklearn.naive_bayes import GaussianNB

clf = GaussianNB()
clf.fit(trn_X, trn_Y)

test_acc, test_f1_mac, test_f1_mic, test_auc = valid_no_model(
    clf.predict(test_X), test_Y
)
print(
    "test_acc:",
    test_acc,
    "f1 macro:",
    test_f1_mac,
    "f1 micro",
    test_f1_mic,
    "auc",
    test_auc,
)

test_acc: 0.795 f1 macro: 0.7961224489795918 f1 micro 0.795 auc 0.874803988419627
